In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from iblatlas.atlas import BrainRegions
from one.api import ONE

from iblatlas.atlas import AllenAtlas
from brainbox.io.one import SessionLoader
from tqdm.notebook import tqdm
import seaborn as sns


from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units
from brainwidemap.bwm_loading import merge_probes

from pathlib import Path
import itertools
import pickle as pkl
import warnings

In [2]:
# to find sessions in which stimulus and choice regions are recorded together

In [11]:
df_decoding_session_stim = pd.read_parquet(
    "../../ibl-partial-info-decomp/data/external/stimside_stage2.pqt"
)
df_decoding_region = pd.read_csv("../../ibl-partial-info-decomp/data/external/region_info.csv")
one = ONE(
    base_url="https://openalyx.internationalbrainlab.org",
    password="international",
    silent=True,
    username="intbrainlab",
)
units_df = bwm_units(one)
bwm_df = bwm_query(one)

Loading bwm_query results from fixtures/2023_12_bwm_release.csv
d16d0b38d392b18c0ce8b615ec89d60d7c901df2eeb3432986b62130af28ef01
Loading bwm_query results from fixtures/2023_12_bwm_release.csv


In [119]:
def find_recorded_pairs(df, units_df, column_one, column_two, min_sessions=10, min_neurons=20):
    stim_decoders = df[df[column_one] == 1]
    choice_decoders = df[df[column_two] == 1]
    stim_regions = stim_decoders["Beryl"]
    choice_regions = choice_decoders["Beryl"]

    unit_counts = units_df.groupby(["eid", "Beryl"]).size().reset_index(name="count")
    valid_eid_regions = unit_counts[unit_counts["count"] >= min_neurons]

    df_stim = valid_eid_regions[valid_eid_regions["Beryl"].isin(stim_regions)]
    df_choice = valid_eid_regions[valid_eid_regions["Beryl"].isin(choice_regions)]
    stim_unique = df_stim[["eid", "Beryl"]].drop_duplicates()
    choice_unique = df_choice[["eid", "Beryl"]].drop_duplicates()
    df_paired = pd.merge(
        stim_unique, choice_unique, on="eid", how="inner", suffixes=("_stim", "_choice")
    )
    df_paired = df_paired[df_paired["Beryl_stim"] != df_paired["Beryl_choice"]]
    pair_counts = (
        df_paired.groupby(["Beryl_stim", "Beryl_choice"])["eid"]
        .agg(
            recording_count="nunique",
            eids_list=lambda x: list(x.unique()),
        )
        .reset_index()
        .sort_values(by="recording_count", ascending=False)
    )
    df_new = pair_counts[pair_counts["recording_count"] >= min_sessions]
    return df_new

In [120]:
df_ephys = find_recorded_pairs(df_decoding_region, units_df, "stim_dec_sig", "choice_dec_sig")

In [121]:
df_ephys

,Beryl_stim,Beryl_choice,recording_count,eids_list
482,SCm,MRN,23,"[07dc4b76-5b93-4a03-82a0-b3d9cc73f412, 111c176..."
361,MRN,SCm,23,"[07dc4b76-5b93-4a03-82a0-b3d9cc73f412, 111c176..."
273,LP,PO,21,"[0802ced5-33a3-405e-8336-b65ebc5cb07c, 0f25376..."
355,MRN,PRNr,13,"[14127fdb-2e66-4823-b124-f49c128ba94d, 16c3667..."
435,PRNr,MRN,13,"[14127fdb-2e66-4823-b124-f49c128ba94d, 16c3667..."
27,APN,MRN,12,"[03d9a098-07bf-4765-88b7-85f8d8f620cc, 1ca83b2..."
331,MRN,APN,12,"[03d9a098-07bf-4765-88b7-85f8d8f620cc, 1ca83b2..."
68,CP,MOp,12,"[03063955-2523-47bd-ae57-f7489dd40f15, 626126d..."
290,LP,VISa,11,"[0802ced5-33a3-405e-8336-b65ebc5cb07c, 0a018f1..."
555,VISa,LP,11,"[0802ced5-33a3-405e-8336-b65ebc5cb07c, 0a018f1..."


In [92]:
df_manifold = find_recorded_pairs(df_decoding_region, units_df, "stim_man_sig", "choice_man_sig")

In [122]:
df_ephys.to_csv("../data/generated/ephys_communication/df_stim_choice_pairs.csv")

In [115]:
df_manifold.to_csv("../data/generated/ephys_communication/df_stim_choice_manifolds.csv")

### Look at ouptut

In [123]:
data = pkl.load(
    open(
        "../data/generated/ephys_communication/02fbb6da-3034-47d6-a61b-7d06c796a830_region_predictions.pkl",
        "rb",
    )
)

In [124]:
data["complete"]["null_r2"]

array([[[0., 0.]]])

In [125]:
data["complete"]["true_r2"]

array([[-0.09099269]])

In [130]:
data["regions_choice"]

['PO']

In [131]:
data["regions_stim"]

['VISa']

In [134]:
data["incongruent"]["true_r2"]

array([[-2.48141839e+30]])